# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, sum as _sum, expr, sha2, concat_ws

# Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.lc_loans_raw")

# Data Transformations

## tạo value cho cột id và memberid

In [0]:
df = df.withColumn("id", expr("uuid()"))

borrower_traits = [
    "emp_title",      
    "annual_inc",     
    "home_ownership", 
    "addr_state"     
]

df = df.withColumn(
    "member_id", 
    sha2(concat_ws("_", *[col(c) for c in borrower_traits]), 256)
)

## Loại bỏ các cột có giá trị null chiếm > 80% 

In [0]:
total_rows = df.count()
threshold = 0.8

null_counts_row = df.select([
    _sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
]).first().asDict()

cols_to_drop = [
    c for c, null_count in null_counts_row.items() 
    if (null_count / total_rows) > threshold
]

print(f"Tổng số dòng: {total_rows}")
print(f"Số lượng cột bị xóa (> {threshold*100}% Null): {len(cols_to_drop)}")
print(f"Danh sách các cột bị xóa:\n{cols_to_drop}")

df_silver = df.drop(*cols_to_drop)